In [1]:
# ── Cell 1: Install + Imports ──
!pip install -q mlflow

import os, random, warnings
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models

import mlflow
import mlflow.pytorch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"GPUs    : {torch.cuda.device_count()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.5/907.5 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 12.0 MB/s eta 0:00:00
Device  : cuda
GPU     : Tesla T4
VRAM    : 15

In [2]:
# ── Cell 2: Data Loading + Preprocessing ──
DATA_DIR  = Path('/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000')
IMG_DIR_1 = DATA_DIR / 'ham10000_images_part_1'
IMG_DIR_2 = DATA_DIR / 'ham10000_images_part_2'

# label map
LABEL_MAP  = {'nv':0, 'mel':1, 'bkl':2, 'bcc':3, 'akiec':4, 'vasc':5, 'df':6}
CLASS_NAMES = list(LABEL_MAP.keys())
NUM_CLASSES = len(CLASS_NAMES)

# load metadata
df = pd.read_csv(DATA_DIR / 'HAM10000_metadata.csv')
df['label'] = df['dx'].map(LABEL_MAP)

# map image_id to full path
def get_image_path(image_id):
    for d in [IMG_DIR_1, IMG_DIR_2]:
        p = d / f"{image_id}.jpg"
        if p.exists(): return str(p)
    return None

df['path'] = df['image_id'].apply(get_image_path)
print(f"Total images : {len(df)}")
print(f"Missing      : {df['path'].isna().sum()}")
print(f"\nClass distribution:")
print(df['dx'].value_counts())

# split 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])
val_df,  test_df  = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])
print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# class weights
class_counts  = df['label'].value_counts().sort_index().values
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = (class_weights / class_weights.sum() * NUM_CLASSES).to(device)
print(f"\nClass weights:")
for name, w in zip(CLASS_NAMES, class_weights):
    print(f"  {name:6s} → {w:.4f}")

# weighted sampler
train_labels   = train_df['label'].values
sample_weights = class_weights.cpu().numpy()[train_labels]
sampler        = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
print(f"\nSampler ready ✅")

Total images : 10015
Missing      : 0

Class distribution:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

Train: 7010 | Val: 1502 | Test: 1503

Class weights:
  nv     → 0.0460
  mel    → 0.2771
  bkl    → 0.2806
  bcc    → 0.6000
  akiec  → 0.9431
  vasc   → 2.1717
  df     → 2.6816

Sampler ready ✅


In [3]:
# ── Cell 3: Augmentation + Dataset + DataLoader ──
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE      = 224

# medium augmentation — proven best
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=360),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, hue=0.1),
    transforms.GaussianBlur(kernel_size=3),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# dataset class
class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = int(row['label'])
        if self.transform:
            image = self.transform(image)
        return image, label

# dataloaders
train_ds = SkinDataset(train_df, transform=train_transform)
val_ds   = SkinDataset(val_df,   transform=val_transform)
test_ds  = SkinDataset(test_df,  transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False,
                          num_workers=2, pin_memory=True)

# sanity check
imgs, labels = next(iter(train_loader))
print(f"Batch shape    : {imgs.shape}")
print(f"Labels sample  : {labels[:8].tolist()}")
print(f"Train batches  : {len(train_loader)}")
print(f"Val   batches  : {len(val_loader)}")
print(f"Test  batches  : {len(test_loader)}")
print(f"\nDataLoaders ready ✅")

Batch shape    : torch.Size([16, 3, 224, 224])
Labels sample  : [3, 0, 3, 3, 3, 3, 5, 5]
Train batches  : 439
Val   batches  : 94
Test  batches  : 94

DataLoaders ready ✅


In [4]:
# ── Cell 4: Model ──
def build_model(num_classes=NUM_CLASSES):
    model = models.efficientnet_b4(weights='IMAGENET1K_V1')
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features, num_classes
    )
    model = model.to(device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        print(f"Using {torch.cuda.device_count()} GPUs ✅")
    return model

model = build_model()
total_params  = sum(p.numel() for p in model.parameters())
train_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params/1e6:.1f}M")
print(f"Trainable params : {train_params/1e6:.1f}M")
print(f"Model ready ✅")

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 145MB/s]


Using 2 GPUs ✅
Total params     : 17.6M
Trainable params : 17.6M
Model ready ✅


In [5]:
# ── Cell 5: Training Loop ──
import time

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels      = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            preds       = outputs.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss   = total_loss / total
    accuracy   = correct / total
    macro_f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    macro_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    mel_recall   = recall_score(all_labels, all_preds, average=None, zero_division=0)[1]
    return avg_loss, accuracy, macro_f1, macro_recall, mel_recall


print("Training loop ready ✅")

Training loop ready ✅


In [6]:
# ── Cell 6: Train + Save Best Model ──
EPOCHS     = 20
best_f1    = 0.0
best_epoch = 0
start_time = time.time()

mlflow.set_experiment('skin_lesion_final')

with mlflow.start_run(run_name='efficientnet_b4_adam_cosine_medium'):
    mlflow.log_params({
        'backbone'  : 'efficientnet_b4',
        'optimizer' : 'adam',
        'loss'      : 'cross_entropy',
        'aug'       : 'medium',
        'scheduler' : 'cosine',
        'epochs'    : EPOCHS,
        'batch_size': 16,
        'lr'        : 1e-3
    })

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc                        = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, val_f1, val_recall, mel_recall = evaluate(model, val_loader, criterion)
        scheduler.step()

        mlflow.log_metrics({
            'train_loss' : round(train_loss,   4),
            'train_acc'  : round(train_acc,    4),
            'val_loss'   : round(val_loss,     4),
            'val_acc'    : round(val_acc,      4),
            'val_f1'     : round(val_f1,       4),
            'val_recall' : round(val_recall,   4),
            'mel_recall' : round(mel_recall,   4),
        }, step=epoch)

        print(
            f"Ep {epoch:02d}/{EPOCHS} | "
            f"train_loss: {train_loss:.4f} | "
            f"val_loss: {val_loss:.4f} | "
            f"val_acc: {val_acc:.4f} | "
            f"val_f1: {val_f1:.4f} | "
            f"mel_recall: {mel_recall:.4f}"
        )

        # save best model
        if val_f1 > best_f1:
            best_f1    = val_f1
            best_epoch = epoch
            torch.save(
                model.state_dict(),
                '/kaggle/working/best_model.pt'
            )
            print(f"  ✅ Best model saved — F1: {best_f1:.4f}")

    elapsed = (time.time() - start_time) / 60
    mlflow.log_metrics({
        'best_val_f1' : round(best_f1, 4),
        'best_epoch'  : best_epoch,
        'train_min'   : round(elapsed, 2)
    })

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Best F1        : {best_f1:.4f}")
print(f"Best epoch     : {best_epoch}")
print(f"Total time     : {elapsed:.1f} min")
print(f"Model saved    : /kaggle/working/best_model.pt")

2026/06/08 09:38:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/08 09:38:30 INFO mlflow.store.db.utils: Updating database tables
2026/06/08 09:38:33 INFO mlflow.tracking.fluent: Experiment with name 'skin_lesion_final' does not exist. Creating a new experiment.


Ep 01/20 | train_loss: 1.0802 | val_loss: 0.8693 | val_acc: 0.6332 | val_f1: 0.5870 | mel_recall: 0.8922
  ✅ Best model saved — F1: 0.5870
Ep 02/20 | train_loss: 0.7285 | val_loss: 0.6695 | val_acc: 0.7423 | val_f1: 0.5995 | mel_recall: 0.7066
  ✅ Best model saved — F1: 0.5995
Ep 03/20 | train_loss: 0.5863 | val_loss: 0.5452 | val_acc: 0.8089 | val_f1: 0.6886 | mel_recall: 0.4731
  ✅ Best model saved — F1: 0.6886
Ep 04/20 | train_loss: 0.5046 | val_loss: 0.6231 | val_acc: 0.7497 | val_f1: 0.7034 | mel_recall: 0.7665
  ✅ Best model saved — F1: 0.7034
Ep 05/20 | train_loss: 0.4411 | val_loss: 0.5658 | val_acc: 0.7883 | val_f1: 0.7283 | mel_recall: 0.6707
  ✅ Best model saved — F1: 0.7283
Ep 06/20 | train_loss: 0.4053 | val_loss: 0.6660 | val_acc: 0.7716 | val_f1: 0.6775 | mel_recall: 0.3353
Ep 07/20 | train_loss: 0.3532 | val_loss: 0.4695 | val_acc: 0.8362 | val_f1: 0.7502 | mel_recall: 0.6467
  ✅ Best model saved — F1: 0.7502
Ep 08/20 | train_loss: 0.3085 | val_loss: 0.6779 | val_acc: 0